In [ ]:
# 1. Install geemap if not exists
!pip install geemap

import ee
import geemap

# 2. Authenticate and Initialize with YOUR NEW Project ID
# This will trigger a popup to login with your new Gmail account
try:
    ee.Initialize(project='wheat-project-483813-485006')
    print("Successfully connected!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='wheat-project-483813-485006')
    print("Successfully connected after authentication!")

# ==========================================
# PART 1: DEFINE 50 WHEAT REGIONS (FIXED)
# ==========================================
print("Generating 50 High-Density Wheat Regions...")

states = ee.FeatureCollection('TIGER/2018/States')\
    .filter(ee.Filter.inList('NAME', ['Kansas', 'Oklahoma', 'Texas', 'Nebraska', 'Colorado']))

# Load USDA Cropland Data Layer (2020)
cdl = ee.Image('USDA/NASS/CDL/2020').select('cropland')
winter_wheat = cdl.eq(24) # Class 24 is Winter Wheat

# Load Counties
counties = ee.FeatureCollection('TIGER/2018/Counties').filterBounds(states)

def get_wheat_sum(feature):
    sum_wheat = winter_wheat.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=feature.geometry(),
        scale=5000,
        maxPixels=1e9
    )
    return feature.set('wheat_pixels', sum_wheat.get('cropland'))

# Get top 50 counties
top_counties = counties.map(get_wheat_sum).sort('wheat_pixels', False).limit(50)

def create_roi(feature):
    center = feature.geometry().centroid()
    geom = center.buffer(5000).bounds()
    # CRITICAL FIX: Wrap the Geometry in ee.Feature(...)
    return ee.Feature(geom).copyProperties(feature)

regions = top_counties.map(create_roi)
print(f"Regions defined. Count: {regions.size().getInfo()}")

# ==========================================
# PART 2: HARMONIZE SATELLITE DATA
# ==========================================
optical_bands = ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']
l5_bands = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']
l8_bands = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']

def prep_l5(image):
    scaled = image.select(l5_bands, optical_bands).multiply(2.75e-05).add(-0.2)
    qa = image.select('QA_PIXEL')
    mask = qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))
    return scaled.updateMask(mask).copyProperties(image, ['system:time_start'])

def prep_l8_9(image):
    scaled = image.select(l8_bands, optical_bands).multiply(2.75e-05).add(-0.2)
    qa = image.select('QA_PIXEL')
    mask = qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))
    return scaled.updateMask(mask).copyProperties(image, ['system:time_start'])

col_l5 = ee.ImageCollection('LANDSAT/LT05/C02/T1_L2').map(prep_l5)
col_l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').map(prep_l8_9)
col_l9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').map(prep_l8_9)

merged_optical = col_l5.merge(col_l8).merge(col_l9)\
    .filterBounds(states)\
    .sort('system:time_start')

# ==========================================
# PART 3: WEATHER DATA (ERA5)
# ==========================================
era5 = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY").select(
    ['temperature_2m', 'total_precipitation', 'surface_net_solar_radiation'],
    ['Temp_K', 'Precip_m', 'Solar_J']
)

# ==========================================
# PART 4: DEKADAL COMPOSITING LOGIC
# ==========================================
# Note: We define the logic here, but we don't run the full date range at once.
# The actual execution happens in the loop below.
n_days = 10

def process_dekad(start_day_ms):
    start = ee.Date(start_day_ms)
    end = start.advance(n_days, 'day')

    # Filter for specific time window
    dekad_col = merged_optical.filterDate(start, end)

    # Logic to run only if images exist
    def compute_metrics():
        optical_img = dekad_col.median()

        weather_col = era5.filterDate(start, end)
        temp_mean = weather_col.select('Temp_K').mean()
        precip_sum = weather_col.select('Precip_m').sum()
        solar_mean = weather_col.select('Solar_J').mean()

        ndvi = optical_img.normalizedDifference(['NIR', 'Red']).rename('NDVI')
        evi = optical_img.expression(
            '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))', {
                'NIR': optical_img.select('NIR'),
                'RED': optical_img.select('Red'),
                'BLUE': optical_img.select('Blue')
            }).rename('EVI')

        combined = ndvi.addBands(evi).addBands(temp_mean).addBands(precip_sum).addBands(solar_mean)

        return combined.reduceRegions(
            collection=regions,
            reducer=ee.Reducer.mean(),
            scale=5000
        ).map(lambda f: f.set('Date', start.format('YYYY-MM-dd')))

    return ee.Algorithms.If(
        dekad_col.size().gt(0),
        compute_metrics(),
        ee.FeatureCollection([])
    )

# ==========================================
# PART 5: THE REVERSE BATCH EXPORT LOOP
# ==========================================
# We use range(2024, 1983, -1) to go: 2024, 2023, ..., 1984
reverse_years = [1987]

print(f"Launching REVERSE Batch Exports (Starting 2024 -> Downwards)...")

for year in reverse_years:
    # Define the chunk window (Exactly 1 year)
    start_chunk = f'{year}-01-01'
    end_chunk = f'{year}-12-31'

    # 1. Generate Dates for JUST this year
    chunk_dates = ee.List.sequence(
        ee.Date(start_chunk).millis(),
        ee.Date(end_chunk).millis(),
        n_days * 24 * 60 * 60 * 1000
    )

    # 2. Process JUST this chunk
    chunk_results = ee.FeatureCollection(chunk_dates.map(process_dekad)).flatten()
    valid_chunk = chunk_results.filter(ee.Filter.notNull(['NDVI']))

    # 3. Create a unique task name
    task_name = f'BIFORGE_Chunk_{year}'

    task = ee.batch.Export.table.toDrive(
        collection=valid_chunk,
        description=task_name,
        folder='BIFORGE_Data', # This will save to your new Gmail's Drive
        fileFormat='CSV',
        selectors=['Date', 'system:index', 'NDVI', 'EVI', 'Temp_K', 'Precip_m', 'Solar_J']
    )

    task.start()
    print(f"-> Started Task: {task_name} (Year: {year})")

print("All tasks launched! Check GEE Task Manager tab.")

Successfully connected!
Generating 50 High-Density Wheat Regions...
Regions defined. Count: 50
Launching REVERSE Batch Exports (Starting 2024 -> Downwards)...
-> Started Task: BIFORGE_Chunk_1987 (Year: 1987)
All tasks launched! Check GEE Task Manager tab.
